# executorlib with SLURM 
[executorlib](https://executorlib.readthedocs.io) extends the `Executor` interface from the Python standard library's [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html) to distribute Python functions as jobs on an HPC cluster. This notebook deliberately submits a job with an incomplete `resource_dict` to show how `SlurmClusterExecutor` surfaces `sbatch` submission failures through the returned `Future`. See the [resource dictionary troubleshooting page](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary) for the required keys.

Based on the [cmti and cmmg clusters](https://docs.mpcdf.mpg.de/doc/computing/clusters/systems/Sustainable_Materials.html) hosted at the MPCDF for the MPI for Sustainable Materials.

In [1]:
import executorlib

In [2]:
executorlib.__version__

'0.0.1'

In [3]:
executorlib.__path__

['/u/janj/projects/executorlib/src/executorlib']

## Submit Python Function to SLURM
This is just like using `sbatch` to submit shell scripts. See the [`SlurmClusterExecutor` documentation](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#slurm) for the full set of options.

In [4]:
def echo(i):
    return i

`echo` is a trivial placeholder function — the point of this notebook is not the function itself but the empty `resource_dict={}` passed to `submit()` below. Required keys such as `partition` are intentionally omitted to trigger a submission failure.

In [5]:
with executorlib.SlurmClusterExecutor() as exe:
    f1 = exe.submit(
        echo,
        resource_dict={})
    result = f1.result()
    print(result)

CalledProcessError: Command '['sbatch', '--parsable', '/cmmc/u/janj/notebooks/2026/2026-06-03-prabhath-exe/executorlib_cache/echoe5873521b7330f831bed941744079e83/run_queue.sh']' returned non-zero exit status 1.

Because `resource_dict` does not specify a `partition`, the generated `sbatch` script has no `--partition` line, and SLURM rejects it with `invalid partition specified: (null)`. executorlib runs `sbatch` as a subprocess and raises the resulting `CalledProcessError` from within `f1.result()` — exactly as if `sbatch` had been called directly on the command line without `--partition`.

Rather than letting `.result()` re-raise the exception, `Future.exception()` returns it directly so it can be inspected without a `try`/`except` block. For a `CalledProcessError` raised by `sbatch`, the `.output` attribute holds the captured `stderr` — here it confirms the missing-partition error. See the [resource dictionary troubleshooting page](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary) for the list of keys `resource_dict` accepts, including `partition`.

In [6]:
excep = f1.exception()
excep.output

'sbatch: error: invalid partition specified: (null)\nsbatch: error: Batch job submission failed: Invalid partition name specified\n'

## Clean up

In [7]:
import os
import shutil

In [8]:
cache_dir = "executorlib_cache"
os.listdir(cache_dir)

['echoe5873521b7330f831bed941744079e83_o.h5',
 'echoe5873521b7330f831bed941744079e83']

In [9]:
shutil.rmtree(cache_dir)